In [5]:
import pandas as pd
import os

# 1. Load Data
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_df = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

print("Files loaded. Starting processing...")

# 2. Ensure IPs are numeric and drop rows with invalid IPs
fraud_df['ip_address'] = pd.to_numeric(fraud_df['ip_address'], errors='coerce')
fraud_df = fraud_df.dropna(subset=['ip_address'])
fraud_df['ip_address'] = fraud_df['ip_address'].astype(int)

ip_df['lower_bound_ip_address'] = ip_df['lower_bound_ip_address'].astype(int)
ip_df['upper_bound_ip_address'] = ip_df['upper_bound_ip_address'].astype(int)

# 3. Sort for merge_asof (MANDATORY)
fraud_df = fraud_df.sort_values('ip_address')
ip_df = ip_df.sort_values('lower_bound_ip_address')

# 4. Perform the merge
print("Mapping IPs to Countries (this should be fast)...")
fraud_df = pd.merge_asof(
    fraud_df, 
    ip_df, 
    left_on='ip_address', 
    right_on='lower_bound_ip_address'
)

# 5. Clean up logic
fraud_df.loc[fraud_df['ip_address'] > fraud_df['upper_bound_ip_address'], 'country'] = "Unknown"
fraud_df['country'] = fraud_df['country'].fillna("Unknown")

# 6. Final save
os.makedirs('../data/processed', exist_ok=True)
fraud_df.to_csv('../data/processed/fraud_data_with_country.csv', index=False)

print("SUCCESS! You can now proceed to the next notebook.")

Files loaded. Starting processing...
Mapping IPs to Countries (this should be fast)...
SUCCESS! You can now proceed to the next notebook.
